In [7]:
import cv2
import numpy as np
from PIL import Image, ImageDraw, ImageFont
import math

# Define constants
WIDTH, HEIGHT = 500, 500
FRAMES = 50

# Colors
STRESS_COLOR = (100, 100, 100)  # Gray
CALM_COLOR = (135, 206, 235)  # Light Blue
WHITE = (255, 255, 255)
BLACK = (0, 0, 0)
GLOW_COLOR = (255, 223, 186)  # Soft glow

try:
    FONT = ImageFont.truetype("arial.ttf", 24)
except:
    FONT = ImageFont.load_default()

# Helper functions
def draw_cloud(draw, x, y, size):
    draw.ellipse([x - size, y - size, x + size, y + size], fill=WHITE)

def draw_glow(draw, center, radius, color, intensity=5):
    for i in range(intensity):
        alpha = max(0, 255 - (i * 50))
        draw.ellipse(
            [
                center[0] - radius - i * 5,
                center[1] - radius - i * 5,
                center[0] + radius + i * 5,
                center[1] + radius + i * 5,
            ],
            fill=color,
        )

def create_frame(progress):
    img = Image.new("RGB", (WIDTH, HEIGHT), STRESS_COLOR)
    draw = ImageDraw.Draw(img)

    # Background transition
    blend = progress / FRAMES
    r = int(STRESS_COLOR[0] * (1 - blend) + CALM_COLOR[0] * blend)
    g = int(STRESS_COLOR[1] * (1 - blend) + CALM_COLOR[1] * blend)
    b = int(STRESS_COLOR[2] * (1 - blend) + CALM_COLOR[2] * blend)
    draw.rectangle([0, 0, WIDTH, HEIGHT], fill=(r, g, b))

    # Clouds movement and fading
    for i in range(5):
        x = (progress * 10 + i * 100) % WIDTH
        y = HEIGHT // 3 + int(20 * math.sin(progress / 5 + i))
        alpha = max(0, 255 - int(255 * blend))
        draw_cloud(draw, x, y, 40 - int(20 * blend))

    # Add some moving stars (particles)
    for i in range(50):
        x = (i * 20 + progress * 3) % WIDTH
        y = HEIGHT // 4 + int(40 * math.cos(i / 5 + progress / 5))
        draw.ellipse([x - 2, y - 2, x + 2, y + 2], fill=WHITE)

    # Person's face
    face_center = (WIDTH // 2, HEIGHT // 2 + 50)
    face_radius = 60
    draw_glow(draw, face_center, face_radius + 10, GLOW_COLOR)
    draw.ellipse([face_center[0] - face_radius, face_center[1] - face_radius,
                  face_center[0] + face_radius, face_center[1] + face_radius], fill=WHITE)

    # Face expression transition
    if progress < FRAMES // 2:
        # Stressed expression (frown)
        draw.arc([face_center[0] - 30, face_center[1] + 10, face_center[0] + 30, face_center[1] + 40],
                 start=20, end=160, fill=BLACK, width=2)
    else:
        # Calm expression (smile)
        draw.arc([face_center[0] - 30, face_center[1] + 10, face_center[0] + 30, face_center[1] + 40],
                 start=200, end=340, fill=BLACK, width=2)

    # Eyes
    eye_offset = 20
    draw.ellipse([face_center[0] - eye_offset - 5, face_center[1] - 10,
                  face_center[0] - eye_offset + 5, face_center[1]], fill=BLACK)
    draw.ellipse([face_center[0] + eye_offset - 5, face_center[1] - 10,
                  face_center[0] + eye_offset + 5, face_center[1]], fill=BLACK)

    # Text with glow effect
    if progress < FRAMES // 2:
        text = "Overwhelmed"
        glow_color = (255, 0, 0)  # Stress glow
    else:
        text = "Relax"
        glow_color = (0, 255, 0)  # Calm glow

    text_bbox = draw.textbbox((0, 0), text, font=FONT)
    text_width = text_bbox[2] - text_bbox[0]
    text_height = text_bbox[3] - text_bbox[1]
    text_position = ((WIDTH - text_width) // 2, HEIGHT // 4)
    draw_glow(draw, (text_position[0] + text_width // 2, text_position[1] + text_height // 2), 30, glow_color)
    draw.text(text_position, text, fill=BLACK, font=FONT)

    # Draw progress bar
    progress_width = int((progress / FRAMES) * WIDTH)
    draw.rectangle([0, HEIGHT - 20, progress_width, HEIGHT], fill=(0, 255, 0))  # Green bar

    return np.array(img)

# Initialize video writer
fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Codec for MP4
video_writer = cv2.VideoWriter('stress_to_calm_interactive.mp4', fourcc, 10, (WIDTH, HEIGHT))

# Generate frames and write to video
for i in range(FRAMES):
    frame = create_frame(i)
    video_writer.write(frame)

# Release the video writer
video_writer.release()

print("MP4 created with interactive features: stress_to_calm_interactive.mp4")


MP4 created with interactive features: stress_to_calm_interactive.mp4
